# Tests: `fasterai.distill.losses` (source `nbs/distill/losses.ipynb`)

In [ ]:
from fastcore.test import *
import torch
import torch.nn as nn
from fasterai.distill.losses import *

In [ ]:
from fastcore.test import *

# Output-based losses return scalars
pred_s, pred_t = torch.randn(4, 10), torch.randn(4, 10)

test_eq(SoftTarget(pred_s, pred_t).dim(), 0)
test_eq(Logits(pred_s, pred_t).dim(), 0)
test_eq(Mutual(pred_s, pred_t).dim(), 0)

# Different temperature → different loss
test_ne(SoftTarget(pred_s, pred_t, T=1), SoftTarget(pred_s, pred_t, T=10))

# Feature-based losses return scalars
fm_s = {'l1': torch.randn(4, 32, 8, 8), 'l2': torch.randn(4, 64, 4, 4)}
fm_t = {'l1': torch.randn(4, 32, 8, 8), 'l2': torch.randn(4, 64, 4, 4)}

test_eq(Attention(fm_s, fm_t).dim(), 0)
test_eq(FitNet(fm_s, fm_t).dim(), 0)

# Identical inputs → ~0 loss
fm_id = {'l1': torch.randn(4, 32, 8, 8)}
test_close(FitNet(fm_id, fm_id).item(), 0.0, eps=1e-5)
test_close(Attention(fm_id, fm_id).item(), 0.0, eps=1e-4)

# All losses non-negative
assert SoftTarget(pred_s, pred_t) >= 0
assert Attention(fm_s, fm_t) >= 0
assert FitNet(fm_s, fm_t) >= 0

# ActivationBoundaries returns scalar
test_eq(ActivationBoundaries(fm_s, fm_t).dim(), 0)
assert ActivationBoundaries(fm_s, fm_t) >= 0

# DecoupledKD tests
target = torch.randint(0, 10, (4,))

# Returns scalar
test_eq(DecoupledKD(pred_s, pred_t, target=target).dim(), 0)

# Non-negative
assert DecoupledKD(pred_s, pred_t, target=target) >= 0

# Different alpha/beta → different loss
loss_a = DecoupledKD(pred_s, pred_t, target=target, alpha=1.0, beta=1.0)
loss_b = DecoupledKD(pred_s, pred_t, target=target, alpha=1.0, beta=8.0)
test_ne(loss_a, loss_b)

# Different temperatures → different loss
test_ne(DecoupledKD(pred_s, pred_t, target=target, T=1),
        DecoupledKD(pred_s, pred_t, target=target, T=10))

# Identical logits → ~0 loss
pred_same = torch.randn(4, 10)
test_close(DecoupledKD(pred_same, pred_same, target=target).item(), 0.0, eps=1e-4)

# Raises ValueError without target
with ExceptionExpected(ValueError):
    DecoupledKD(pred_s, pred_t)

# Works with 2 classes (minimal case)
pred_2 = torch.randn(4, 2)
target_2 = torch.randint(0, 2, (4,))
test_eq(DecoupledKD(pred_2, pred_2, target=target_2).dim(), 0)

# --- NKD mode (normalize=True) ---

# Returns scalar, non-negative
test_eq(DecoupledKD(pred_s, pred_t, target=target, normalize=True).dim(), 0)
assert DecoupledKD(pred_s, pred_t, target=target, normalize=True) >= 0

# Identical logits → ~0 loss (NKD mode)
test_close(DecoupledKD(pred_same, pred_same, target=target, normalize=True).item(), 0.0, eps=1e-4)

# Works with 2 classes in NKD mode
test_eq(DecoupledKD(pred_2, pred_2, target=target_2, normalize=True).dim(), 0)

# Extreme confidence: no nan/inf (reviewer catch)
pred_extreme = torch.tensor([[100., 0., 0., 0.], [0., 100., 0., 0.]])
target_extreme = torch.tensor([0, 1])
loss_ext = DecoupledKD(pred_extreme, pred_extreme, target=target_extreme)
assert torch.isfinite(loss_ext), f"DKD produced {loss_ext} with extreme logits"
loss_ext_nkd = DecoupledKD(pred_extreme, pred_extreme, target=target_extreme, normalize=True)
assert torch.isfinite(loss_ext_nkd), f"NKD produced {loss_ext_nkd} with extreme logits"

In [ ]:
#| slow
# Integration test: DecoupledKD (both modes) with KnowledgeDistillationCallback
from torch.utils.data import TensorDataset
from fastai.data.core import DataLoaders
from fastai.learner import Learner
from fasterai.distill.distillation_callback import KnowledgeDistillationCallback
from functools import partial

_teacher = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(16, 10))
_student = nn.Sequential(nn.Conv2d(3, 8, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(8, 10))
_X = torch.randn(64, 3, 8, 8)
_y = torch.randint(0, 10, (64,))
_dls = DataLoaders.from_dsets(TensorDataset(_X[:48], _y[:48]), TensorDataset(_X[48:], _y[48:]), bs=16, device='cpu')

# Test standard DKD mode
_cb = KnowledgeDistillationCallback(teacher=_teacher, loss=DecoupledKD)
_learn = Learner(_dls, _student, loss_func=nn.CrossEntropyLoss(), cbs=[_cb])
_learn.fit(2)

# Test NKD mode (normalize=True)
_student2 = nn.Sequential(nn.Conv2d(3, 8, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(8, 10))
_cb2 = KnowledgeDistillationCallback(teacher=_teacher, loss=partial(DecoupledKD, normalize=True))
_learn2 = Learner(_dls, _student2, loss_func=nn.CrossEntropyLoss(), cbs=[_cb2])
_learn2.fit(2)